# Tramo lineal de la rampa: tres criterios, y qué tan bien cierran la ida y la vuelta

**Qué se quiere mostrar.** La platina barre un triángulo: sube (ida) y baja (vuelta).
Sólo una parte de cada pasada es una rampa a velocidad constante; el resto es
aceleración, frenado y las puntas. Esa parte lineal es la única en la que un scan sirve
para medir. Las preguntas del cuaderno son tres:

1. **¿Dónde empieza y dónde termina la parte lineal de cada pasada?** Primero se separa
   la ida de la vuelta (arranque, pico y fin, leídos del comando) y recién después, *dentro
   de cada pata*, se busca el tramo lineal. Se lo busca de tres maneras independientes
   (celdas 2, 3 y 4). Que las tres coincidan es la prueba de que el tramo es real y no lo
   está poniendo el umbral que elegimos.
2. **¿La ida y la vuelta barren el mismo tramo de espacio?** No: el lazo se atrasa una
   cantidad fija `v·tau`, y como en la vuelta la velocidad cambia de signo, los dos tramos
   quedan corridos `2·v·tau` entre sí. Ese **desfasaje** es lo que se pierde si se quieren
   usar las dos pasadas sobre la misma grilla. Se mide entre los dos puntos que
   corresponden a la **misma posición comandada** (el centro del solapamiento), no entre
   las puntas de cada tramo: las puntas dependen del criterio, el desfasaje no.
3. **¿Qué WTR y qué SUD conviene usar?** (celdas 6 y 7). WTR fija la velocidad; SUD,
   cuántos puntos se gastan acelerando en cada punta.

**Los tres criterios, en una línea cada uno.** Los tres son el mismo tipo de pregunta con
señales distintas, y los tres usan una **tolerancia relativa del 2 %** sobre el valor
estable, no un percentil del recorrido:

| celda | criterio | señal | de dónde sale |
|---|---|---|---|
| 2 · A | velocidad constante al 2 % | posición **comandada** derivada | lo que hace el generador |
| 3 · B | error de seguimiento en su meseta al 2 % | `error_um` del **controlador** | señal medida, independiente |
| 4 · C | el residuo a una recta no supera `max(3σ, dx/2)` | posición **medida** | lo que hace la platina |

Cada una de esas celdas muestra **dos figuras y nada más**: la determinación del tramo
(la rampa con ida y vuelta separadas, y abajo la señal con su banda de tolerancia) y,
aparte, el **solapamiento** de los dos tramos sobre el eje de posición.

> Ejecutar las celdas en orden. En Google Colab la celda 0 clona el repositorio con los
> datos; en local encuentra la carpeta sola.

## Celda 0 — lo necesario

Imports, acceso a los datos y las funciones que se repiten en todas las celdas:

- `cargar` — una corrida (posición comandada, medida, error del controlador, `dx` medido);
- `separar_patas` / `patas` — **lo único que separa la ida de la vuelta**: arranque, pico y
  fin del triángulo, leídos del comando;
- `derivar` — velocidad, promediando sobre un peldaño entero de la escalera del generador;
- `meseta_por_tolerancia` — el criterio del 2 % (celdas 2 y 3);
- `ruido` — el ruido de sensado, medido con la platina quieta (celda 4);
- `resumen` / `reporte` — los mismos números para los tres métodos;
- `fig_tramo` / `fig_solapamiento` — las dos figuras, siempre iguales.

In [ ]:
# =============================================================================
#  CELDA 0 - lo necesario: acceso a los datos y las funciones que se repiten
# =============================================================================
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def buscar_repo():
    """Carpeta del repositorio. En Colab lo clona; en local lo busca hacia arriba."""
    for d in (Path.cwd(), *Path.cwd().parents):
        if (d / "datos/raw").is_dir():
            return d
    destino = Path.cwd() / "laboratorio-7"
    if not (destino / "datos/raw").is_dir():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/franaguirreram/laboratorio-7.git",
                        str(destino)], check=True)
    return destino


REPO = buscar_repo()
RAW, META = REPO / "datos/raw", REPO / "datos/metadata"

T_SERVO_US = 39.999984          # [MEDIDO] periodo del lazo de servo del E-517

plt.rcParams.update({"font.family": "serif", "font.size": 9, "figure.dpi": 120,
                     "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5})
COL = {"ida": "#1f77b4", "vuelta": "#d62728", "comun": "#2ca02c",
       "cmd": "0.75", "med": "0.25"}


# --------------------------------------------------------------- lectura ---
def leer_meta(stem):
    """Metadata de una corrida, como diccionario de strings."""
    lineas = (META / f"{stem}_metadata.txt").read_text().splitlines()
    return dict(l.split("=", 1) for l in lineas if "=" in l and not l.startswith("#"))


def separar_patas(x_cmd):
    """Indices de arranque, pico y fin del triangulo, leidos del COMANDO.

    Esto es lo unico que separa la IDA de la VUELTA: todo lo demas se calcula
    despues, dentro de cada pata por separado.
    """
    excursion = x_cmd.max() - x_cmd.min()
    arranque = int(np.argmax(np.abs(x_cmd - x_cmd[0]) > excursion / 1000))
    pico = int(np.argmax(x_cmd))
    fin = int(np.argmin(np.abs(x_cmd[pico:] - x_cmd[arranque]))) + pico
    return arranque, pico, fin


def cargar(stem):
    """Una corrida completa, con todo lo que despues se usa en todas las celdas."""
    m = leer_meta(stem)
    df = pd.read_csv(RAW / f"{stem}.csv")
    rtr, wtr = int(m["rtr"]), int(m["wtr"])
    x_cmd = df["target_um"].to_numpy()
    x_med = df["current_um"].to_numpy()
    arranque, pico, fin = separar_patas(x_cmd)

    r = {"stem": stem,
         "t_ms": np.arange(len(df)) * rtr * T_SERVO_US / 1e3,
         "x_cmd": x_cmd,
         "x_med": x_med,
         # el error que devuelve el CONTROLADOR, no uno recalculado por nosotros
         "err_nm": df["error_um"].to_numpy() * 1e3,
         "dt_ms": rtr * T_SERVO_US / 1e3,
         "wtr": wtr, "rtr": rtr, "sud": int(m["speedupdown"]),
         "n_total": int(m["n_total"]), "amplitud_um": float(m["amplitud_um"]),
         "arranque": arranque, "pico": pico, "fin": fin,
         "muestras_por_punto": max(wtr // rtr, 1),
         "dwell_us": wtr * T_SERVO_US}

    # dx se MIDE en el medio de la ida: con sud > 0 el generador reparte los
    # puntos de forma no uniforme y 2A/n_total ya no vale.
    a, b = arranque, pico
    centro = x_cmd[a + (b - a) // 3: a + 2 * (b - a) // 3]
    paso = centro[::r["muestras_por_punto"]]
    r["dx_nm"] = float(np.median(np.abs(np.diff(paso))) * 1e3) if len(paso) > 2 else np.nan
    r["v_um_s"] = r["dx_nm"] / r["dwell_us"] * 1e3          # nm/us -> um/s
    return r


def patas(r):
    """Los indices de la ida y de la vuelta, por separado."""
    return {"ida": np.arange(r["arranque"], r["pico"] + 1),
            "vuelta": np.arange(r["pico"] + 1, r["fin"] + 1)}


# ------------------------------------------------------------ herramientas --
def suavizar(y, n):
    """Media movil de n muestras, extendiendo los bordes (no rellena con ceros)."""
    n = max(int(n), 1)
    if n <= 1:
        return np.asarray(y, float)
    pad = n // 2
    yp = np.pad(np.asarray(y, float), pad, mode="edge")
    return np.convolve(yp, np.ones(n) / n, mode="same")[pad:pad + len(y)]


def derivar(t_ms, x, n):
    """Velocidad en um/s. Suaviza antes y despues: el comando es una ESCALERA.

    El generador cambia el setpoint una vez cada WTR ciclos de servo y se queda
    quieto en el medio, asi que derivar punto a punto da cero la mayor parte del
    tiempo. Hay que promediar sobre al menos un peldano entero.
    """
    v = np.gradient(suavizar(x, 2 * n), np.asarray(t_ms) * 1e-3)
    return suavizar(v, n)


def tramo_mas_largo(mascara):
    """Indices del tramo contiguo mas largo de una mascara booleana."""
    idx = np.where(mascara)[0]
    if len(idx) == 0:
        return idx
    return max(np.split(idx, np.where(np.diff(idx) > 1)[0] + 1), key=len)


def meseta_por_tolerancia(r, senal, tol):
    """Tramo de cada pata donde `senal` esta a menos de `tol` de su valor estable.

    El criterio es un ERROR RELATIVO, no un percentil del recorrido: en cada
    pata se toma como valor estable s0 la mediana de la mitad central (que
    seguro cae en la parte recta) y se acepta mientras |s - s0| <= tol*|s0|.
    De esa mascara se queda el tramo contiguo mas largo.

    Devuelve los indices por pata y el valor estable de cada una.
    """
    seg, estable = {}, {}
    for cual, idx in patas(r).items():
        centro = idx[len(idx) // 4: 3 * len(idx) // 4]
        s0 = float(np.median(senal[centro]))
        dentro = np.abs(senal[idx] - s0) <= tol * abs(s0)
        seg[cual] = idx[tramo_mas_largo(dentro)]
        estable[cual] = s0
    return seg, estable


def zona_quieta(r):
    """Muestras con la platina parada: el ultimo cuarto de la grabacion.

    La ventana grabada es mas larga que la onda, asi que despues del ultimo punto
    de la wave quedan decenas de ms con la platina quieta en el punto de partida.
    """
    n = len(r["t_ms"])
    return np.arange(n - n // 4, n)


def ruido(r):
    """El ruido de sensado, medido con la platina parada.

    - rms        : dispersion de la posicion medida (incluye la deriva lenta)
    - sigma      : ruido punto a punto, de la diferencia entre muestras
                   consecutivas: var(x[n]-x[n-1]) = 2*sigma^2
    - mediana_dif: la mediana de |diferencias|. Es la estimacion robusta, pero
                   el recorder entrega la posicion con 0.1 nm de resolucion, asi
                   que con la platina quieta satura. Por eso la celda 4 se queda
                   con la mayor de las dos estimaciones.
    """
    z = zona_quieta(r)
    d = np.diff(r["x_med"][z])
    return {"n": len(z), "ms": len(z) * r["dt_ms"],
            "rms_nm": float(np.std(r["x_med"][z]) * 1e3),
            "sigma_nm": float(np.std(d) / np.sqrt(2) * 1e3),
            "mediana_dif_nm": float(np.median(np.abs(d)) * 1e3),
            "resolucion_nm": 0.1}


# ------------------------------------------------- reporte comun y figuras ---
def resumen(r, seg, metodo=""):
    """Los numeros de un tramo lineal: los mismos para los tres metodos.

    duracion, tiempo perdido antes, puntos utilizables, rango de posicion
    barrido, solapamiento de la ida con la vuelta y desfasaje entre las dos.
    """
    t, xm, xc = r["t_ms"], r["x_med"], r["x_cmd"]
    arranca = {"ida": r["arranque"], "vuelta": r["pico"]}
    info = {}
    for cual in ("ida", "vuelta"):
        i = np.asarray(seg.get(cual, []), int)
        if len(i) < 3:
            continue
        x0, x1 = xm[i[0]], xm[i[-1]]
        info[cual] = {
            "i": i,
            "t0": t[i[0]], "t1": t[i[-1]], "dur_ms": t[i[-1]] - t[i[0]],
            "t_antes_ms": t[i[0]] - t[arranca[cual]],
            "n_muestras": len(i),
            "n_puntos": len(i) / r["muestras_por_punto"],
            "lo": min(x0, x1), "hi": max(x0, x1),
            "largo_nm": abs(x1 - x0) * 1e3,
            "v_um_s": float(np.polyfit(t[i] * 1e-3, xm[i], 1)[0]),
            "lag_nm": float(np.median(xm[i] - xc[i])) * 1e3}

    res = {"metodo": metodo, "stem": r["stem"], "tramos": info, "dx_nm": r["dx_nm"]}
    if len(info) == 2:
        lo = max(info["ida"]["lo"], info["vuelta"]["lo"])
        hi = min(info["ida"]["hi"], info["vuelta"]["hi"])
        res["comun_lo"], res["comun_hi"] = lo, hi
        res["solapamiento_nm"] = max(hi - lo, 0.0) * 1e3
        # EL DESFASAJE se mide entre los dos puntos que corresponden a la MISMA
        # posicion comandada, no entre las puntas de los tramos (que empiezan y
        # terminan donde cada criterio los corta). La referencia es el centro
        # del solapamiento. Esa distancia es el 2*v*tau.
        x_ref = (lo + hi) / 2
        marca = {}
        for cual, d in info.items():
            j = d["i"][int(np.argmin(np.abs(xc[d["i"]] - x_ref)))]
            marca[cual] = float(xm[j])
        res["x_ref"], res["marca"] = x_ref, marca
        res["desfasaje_nm"] = (marca["vuelta"] - marca["ida"]) * 1e3
        res["puntos_totales"] = sum(d["n_puntos"] for d in info.values())
        res["puntos_comunes"] = res["solapamiento_nm"] / r["dx_nm"]
        res["t_inversion_ms"] = info["vuelta"]["t0"] - info["ida"]["t1"]
        # tau del lazo, de la propia geometria: desfasaje = 2*v*tau
        v = np.mean([abs(d["v_um_s"]) for d in info.values()])
        res["tau_ms"] = abs(res["desfasaje_nm"]) / 1e3 / (2 * v) * 1e3
    return res


def reporte(res, r):
    """Las seis lineas que hacen falta, y ninguna mas."""
    print(f"{res['metodo']}   ·   {res['stem']}")
    for cual, d in res["tramos"].items():
        print(f"   {cual:>6}  {d['t0']:7.2f} .. {d['t1']:7.2f} ms  ({d['dur_ms']:6.2f} ms, "
              f"{d['n_puntos']:5.0f} puntos)   v = {d['v_um_s']:+9.2f} um/s   "
              f"pierde {d['t_antes_ms']:5.2f} ms al arrancar")
    if "solapamiento_nm" in res:
        print(f"   solapamiento ida ∩ vuelta : {res['comun_lo']:.4f} .. {res['comun_hi']:.4f} um"
              f"  =  {res['solapamiento_nm']:7.1f} nm  ({res['puntos_comunes']:.0f} puntos "
              f"de {res['puntos_totales']:.0f})")
        print(f"   desfasaje ida ↔ vuelta    : {res['desfasaje_nm']:+7.1f} nm   "
              f"→ tau = desfasaje/2v = {res['tau_ms']:.1f} ms")


def fig_tramo(r, res, senal, etiqueta, bandas=(), titulo=""):
    """(a) la rampa con la ida y la vuelta separadas y el tramo lineal encima,
    (b) la senal sobre la que se decidio el tramo, con su banda de tolerancia."""
    t = r["t_ms"]
    fig, ax = plt.subplots(2, 1, figsize=(7.8, 4.8), sharex=True,
                           gridspec_kw=dict(height_ratios=[1.25, 1], hspace=0.12))
    ax[0].plot(t, r["x_cmd"], color=COL["cmd"], lw=1.6, label="comandada")
    for cual, idx in patas(r).items():
        ax[0].plot(t[idx], r["x_med"][idx], color=COL[cual], lw=0.7, alpha=0.45)
    for cual, d in res["tramos"].items():
        ax[0].plot(t[d["i"]], r["x_med"][d["i"]], color=COL[cual], lw=2.4,
                   label=f"{cual}: tramo lineal")
    ax[0].set_ylabel("x [um]")
    ax[0].legend(frameon=False, ncol=3, fontsize=8, loc="lower right")
    ax[0].set_title(titulo, fontsize=9, loc="left")

    ax[1].plot(t, senal, color="k", lw=0.7)
    for y, c in bandas:
        ax[1].axhline(y, color=c, ls="--", lw=0.8)
    for cual, d in res["tramos"].items():
        ax[1].axvspan(t[d["i"][0]], t[d["i"][-1]], color=COL[cual], alpha=0.15, lw=0)
    ax[1].set_ylabel(etiqueta)
    ax[1].set_xlabel("t [ms]")
    return fig, ax


def fig_solapamiento(r, res, titulo="solapamiento de la ida y la vuelta"):
    """Los dos tramos sobre el eje de POSICION: cuanto barren en comun."""
    fig, ax = plt.subplots(figsize=(7.8, 1.9))
    for k, cual in enumerate(("ida", "vuelta")):
        d = res["tramos"].get(cual)
        if d is None:
            continue
        y = 1 - k
        ax.plot([d["lo"], d["hi"]], [y, y], color=COL[cual], lw=8,
                solid_capstyle="butt")
        ax.annotate("", xy=(d["hi"] if cual == "ida" else d["lo"], y - 0.30),
                    xytext=(d["lo"] if cual == "ida" else d["hi"], y - 0.30),
                    arrowprops=dict(arrowstyle="-|>,head_width=0.22,head_length=0.5",
                                    color=COL[cual], lw=1.1, shrinkA=0, shrinkB=0))
    if res.get("solapamiento_nm", 0) > 0:
        ax.axvspan(res["comun_lo"], res["comun_hi"], color=COL["comun"], alpha=0.15, lw=0)
        ax.text((res["comun_lo"] + res["comun_hi"]) / 2, -0.68,
                f"comun a las dos pasadas: {res['solapamiento_nm']:.0f} nm "
                f"({res['puntos_comunes']:.0f} puntos)",
                ha="center", fontsize=8, color=COL["comun"])
        xi, xv = res["marca"]["ida"], res["marca"]["vuelta"]
        for cual, xx in res["marca"].items():
            ax.plot([xx], [1 if cual == "ida" else 0], "|", ms=16, mew=1.6, color="k")
        ax.annotate("", xy=(xv, 1.45), xytext=(xi, 1.45),
                    arrowprops=dict(arrowstyle="<|-|>,head_width=0.2,head_length=0.4",
                                    color="k", lw=1.0, shrinkA=0, shrinkB=0))
        ax.text((xi + xv) / 2, 1.56, f"desfasaje = {res['desfasaje_nm']:+.0f} nm",
                ha="center", fontsize=8.5)
        for x in (xi, xv):
            ax.axvline(x, color="0.6", lw=0.5, ls=":", ymin=0.05, ymax=0.92)
    ax.set_yticks([1, 0])
    ax.set_yticklabels(["ida", "vuelta"])
    ax.set_ylim(-0.9, 1.9)
    ax.set_xlabel("x [um]")
    ax.set_title(titulo, fontsize=9, loc="left")
    return fig, ax


print(f"repositorio : {REPO}")
print(f"corridas    : {len(list(RAW.glob('E517_rampa_*.csv')))} archivos de rampa en datos/raw")

## Celda 1 — todas las mediciones de rampa, numeradas

Se listan las corridas en modo rampa que terminaron bien. La columna **`#`** es el número
de cada medición: es el que se escribe en `IDX` en la celda 5 para analizar cualquiera de
ellas.

Cada corrida queda descripta por cuatro parámetros: **amplitud** (el campo recorrido),
**n_total** (cuántos puntos tiene la wave; con la amplitud fija `dx`, el tamaño del
pixel), **WTR** (cuántos ciclos de servo de 40 µs dura cada punto → el dwell, y con `dx`,
la velocidad) y **SUD** (cuántos puntos de cada punta se gastan acelerando y frenando).

In [ ]:
# =============================================================================
#  CELDA 1 - todas las mediciones de rampa utiles, cada una con su NUMERO
# =============================================================================
# Util = modo rampa, la corrida termino bien (ok=True) y el CSV existe.
# La columna "#" es el numero de cada medicion: es el que se escribe en IDX en
# la celda 5 para analizar cualquiera de ellas.

COLUMNAS = {"t_ms", "target_um", "current_um", "error_um"}

filas = []
for csv in sorted(RAW.glob("E517_rampa_*.csv")):
    try:
        m = leer_meta(csv.stem)
        cabecera = set(csv.open().readline().strip().split(","))
    except OSError:               # falta la metadata, o el archivo no se pudo leer
        continue
    if m.get("modo") != "rampa" or m.get("ok") != "True":
        continue
    # las corridas de diafonia grabaron dos ejes y no guardaron el error del
    # controlador: tienen otras columnas y no sirven para este analisis
    if not COLUMNAS <= cabecera:
        continue
    filas.append({"stem": csv.stem,
                  "amplitud_um": float(m["amplitud_um"]),
                  "n_total": int(m["n_total"]),
                  "wtr": int(m["wtr"]),
                  "rtr": int(m["rtr"]),
                  "sud": int(m["speedupdown"]),
                  "v_um_s": float(m.get("v_um_s") or "nan"),
                  "familia": csv.stem.split("_20")[0]})

RAMPAS = pd.DataFrame(filas).sort_values(
    ["amplitud_um", "n_total", "wtr", "sud", "stem"]).reset_index(drop=True)
RAMPAS["dwell_us"] = RAMPAS.wtr * T_SERVO_US
RAMPAS["dx_nominal_nm"] = 2 * RAMPAS.amplitud_um * 1e3 / RAMPAS.n_total
RAMPAS.index.name = "#"

print(f"{len(RAMPAS)} corridas de rampa utiles · WTR presentes: "
      f"{sorted(RAMPAS.wtr.unique())} · SUD presentes: {sorted(RAMPAS.sud.unique())}\n")
display(RAMPAS[["stem", "amplitud_um", "n_total", "wtr", "rtr", "sud", "v_um_s"]])

## Celda 1b — qué hay medido de cada cosa

La celda que agregaste, adaptada a la tabla nueva: cuántas corridas hay por cada valor de
**WTR**, de **SUD** y de **amplitud**, y después por configuración completa. Es la que
contesta *"¿tengo mediciones con WTR > 20?"* — sí, hay WTR de 5, 10, 20, 40 y 80, y la
celda 6 usa justamente esa familia de cinco velocidades.

In [ ]:
# =============================================================================
#  CELDA 1b - que hay medido de cada cosa
# =============================================================================
# Cuantas corridas hay por cada valor de WTR, de SUD y de amplitud, y despues por
# configuracion completa. Sirve para saber que familias se pueden comparar antes
# de mirar las celdas 6 y 7.

for parametro in ("wtr", "sud", "amplitud_um"):
    print(f"corridas por {parametro}:")
    display(RAMPAS[parametro].value_counts().sort_index()
            .rename_axis(parametro).reset_index(name="corridas"))

por_configuracion = (RAMPAS.groupby(["amplitud_um", "n_total", "wtr", "sud"],
                                    as_index=False)
                     .agg(corridas=("stem", "size"),
                          familias=("familia", lambda s: ", ".join(sorted(set(s)))))
                     .sort_values(["amplitud_um", "n_total", "wtr", "sud"]))
print(f"{len(por_configuracion)} configuraciones distintas:")
display(por_configuracion)

## Celda 2 — método A: primero ida y vuelta, después el tramo lineal por la velocidad

Dos pasos, en este orden:

1. **Separar la ida de la vuelta.** Del comando salen el arranque, el pico y el fin del
   triángulo. De acá en adelante cada pata se trata por separado.
2. **Dentro de cada pata, el tramo lineal sale de la velocidad**, con un criterio de
   **error relativo del 2 %**: se toma como valor estable `v0` la mediana de la velocidad
   en la mitad central de la pata (que seguro cae en la parte recta) y se acepta el tramo
   contiguo más largo donde `|v − v0| ≤ 0.02·|v0|`. Es un umbral con unidades físicas —
   *"la velocidad no se aparta más del 2 % de la que tiene en el medio de la pasada"* — y
   no un percentil del recorrido min–max, que dependía de las puntas.

El criterio se aplica sobre la velocidad **comandada**, que es exacta y sin ruido (la
medida se dibuja en gris: a esta escala es demasiado rugosa para umbralarla al 2 %).

La figura (a)–(b) es **la determinación del tramo**; el solapamiento va **aparte**, en la
figura (c).

> Ojo con lo que este método mide y lo que no: dice dónde el **generador** va a velocidad
> constante, y eso arranca casi enseguida. La platina todavía está enganchando: por eso la
> velocidad ajustada sobre la posición medida en ese tramo da varios por ciento menos que
> `v0`. Los métodos B y C sí ven el enganche, y ahí está la diferencia entre los tres.

In [ ]:
# =============================================================================
#  CELDA 2 - metodo A: ida y vuelta, y el tramo lineal por la VELOCIDAD
# =============================================================================
# Dos pasos, en este orden:
#   1. separar la IDA de la VUELTA, leyendo arranque / pico / fin del comando;
#   2. dentro de cada pata, quedarse con el tramo donde la velocidad esta a
#      menos de TOL de su valor estable. El criterio es un error relativo sobre
#      la meseta, no un percentil del recorrido: "la velocidad no se aparta mas
#      del 2 % de la que tiene en el medio de la pasada".
# Recien despues se mira el solapamiento de los dos tramos.
CORRIDA = "E517_rampa_scan_w10_sud0_20260907_172911_r0"
TOL = 0.02       # 2 % sobre el valor estable de la velocidad

r = cargar(CORRIDA)
print(f"{CORRIDA}\n  amplitud {r['amplitud_um']} um · n_total {r['n_total']} · "
      f"WTR {r['wtr']} · RTR {r['rtr']} · SUD {r['sud']} · dx {r['dx_nm']:.2f} nm · "
      f"v nominal {r['v_um_s']:.2f} um/s\n")

# la velocidad se calcula sobre la posicion COMANDADA (exacta y sin ruido); la
# medida se dibuja en gris como control de que el tramo no lo pone el suavizado
v_cmd = derivar(r["t_ms"], r["x_cmd"], r["muestras_por_punto"])
v_med = derivar(r["t_ms"], r["x_med"], r["muestras_por_punto"])

seg_A, v0_A = meseta_por_tolerancia(r, v_cmd, TOL)
res_A = resumen(r, seg_A, f"METODO A · velocidad dentro del {TOL:.0%} de su valor estable")
reporte(res_A, r)
print(f"   velocidad estable: ida {v0_A['ida']:+.2f} um/s · vuelta {v0_A['vuelta']:+.2f} um/s"
      f"   (banda aceptada: ±{TOL*abs(v0_A['ida']):.2f} um/s)")

# --- figura 1: como se determina el tramo lineal -----------------------------
bandas = [(v0_A[c] * (1 + s * TOL), COL[c]) for c in ("ida", "vuelta") for s in (+1, -1)]
fig, ax = fig_tramo(r, res_A, v_cmd, "v comandada [um/s]", bandas=bandas,
                    titulo=f"(a) {CORRIDA}: ida y vuelta, y el tramo lineal de cada una")
ax[1].plot(r["t_ms"], v_med, color="0.6", lw=0.4, zorder=0)
ax[1].set_title(f"(b) el criterio: |v - v_estable| ≤ {TOL:.0%}  ·  gris: v medida",
                fontsize=8.5, loc="left")
plt.show()

# --- figura 2: el solapamiento, aparte ---------------------------------------
fig_solapamiento(r, res_A, "(c) los dos tramos sobre el eje de posicion")
plt.show()

## Celda 3 — método B: el mismo criterio, sobre el error que mide el controlador

El método A usa el comando, que es una cuenta nuestra. Éste usa una señal **medida e
independiente**: la columna `error_um` que devuelve el recorder del E-517.

En una rampa el lazo se atrasa una cantidad fija, `e = −v·tau`: el error crece hasta una
meseta durante la ida, cruza en el pico y baja hasta la meseta opuesta durante la vuelta.
**Esas dos mesetas son los dos tramos lineales.**

El umbral es el mismo 2 % del método A, ahora sobre la meseta del error (antes era el
90 % del recorrido, que era mucho más flojo y dejaba entrar el arranque). Con el 2 % el
tramo empieza recién cuando la platina ya enganchó: la velocidad ajustada sobre la
posición medida coincide con la nominal, cosa que con el método A no pasaba.

De yapa, la altura de la meseta da `tau = |e|/v` directamente.

In [ ]:
# =============================================================================
#  CELDA 3 - metodo B: el tramo lineal sale del ERROR QUE MIDE EL CONTROLADOR
# =============================================================================
# El metodo A usa el comando, que es una cuenta nuestra. Este usa una senal
# MEDIDA e independiente: la columna error_um del recorder del E-517.
# En una rampa el lazo se atrasa una cantidad fija, e = -v*tau: el error crece
# hasta una meseta en la ida, cruza en el pico y baja hasta la meseta opuesta en
# la vuelta. Esas dos mesetas SON los dos tramos lineales, y se las busca con el
# mismo criterio de la celda 2: |e - e_estable| <= TOL_E * |e_estable|.
TOL_E = 0.02      # mismo 2 % que el metodo A, ahora sobre la meseta del error

e_s = suavizar(r["err_nm"], r["muestras_por_punto"])
seg_B, e0_B = meseta_por_tolerancia(r, e_s, TOL_E)
res_B = resumen(r, seg_B, f"METODO B · error del controlador dentro del {TOL_E:.0%}")
reporte(res_B, r)
print(f"   meseta del error: ida {e0_B['ida']:+.1f} nm · vuelta {e0_B['vuelta']:+.1f} nm"
      f"   →  tau = |e|/v = {abs(e0_B['ida'])/1e3/abs(res_B['tramos']['ida']['v_um_s'])*1e3:.1f} ms")
print(f"   contra el metodo A: ida {res_B['tramos']['ida']['t0']-res_A['tramos']['ida']['t0']:+.2f} / "
      f"{res_B['tramos']['ida']['t1']-res_A['tramos']['ida']['t1']:+.2f} ms en los bordes; "
      f"desfasaje {res_A['desfasaje_nm']:+.0f} nm (A) vs {res_B['desfasaje_nm']:+.0f} nm (B)")

bandas = [(e0_B[c] * (1 + s * TOL_E), COL[c]) for c in ("ida", "vuelta") for s in (+1, -1)]
fig, ax = fig_tramo(r, res_B, e_s, "error del controlador [nm]", bandas=bandas,
                    titulo=f"(a) {CORRIDA}: tramo lineal por el error medido")
ax[1].plot(r["t_ms"], r["err_nm"], color="0.7", lw=0.35, zorder=0)
ax[1].set_title(f"(b) el criterio: |e - e_estable| ≤ {TOL_E:.0%}  ·  gris: error crudo",
                fontsize=8.5, loc="left")
plt.show()

fig_solapamiento(r, res_B, "(c) solapamiento segun el metodo B")
plt.show()

## Celda 4 — método C: hasta dónde una recta describe a la posición medida

Los dos anteriores preguntan *"¿la velocidad (o el error) está en su meseta?"*. Éste
pregunta directamente lo que importa: **¿hasta dónde la posición medida es una recta en el
tiempo?**

Se ajusta una recta y se acepta el tramo mientras el residuo no supere un criterio que
**no se elige a mano**, sale de la propia medición:

- **`3σ` de sensado**, con σ medida en la zona quieta del final de la grabación. Se toma la
  mayor entre la estimación robusta `mediana(|Δx|)/(0.6745·√2)` y el desvío estándar de
  esas mismas diferencias, porque el recorder entrega la posición con 0.1 nm de resolución
  y con la platina quieta la mediana satura;
- **`dx/2`**: la comandada es una **escalera** de `dx` nm, así que aunque el seguimiento
  fuera perfecto el residuo contra la recta oscilaría ±dx/2 **por construcción**.

El criterio adoptado es el mayor de los dos, `max(3σ, dx/2)`: por debajo de `dx/2` no se
estaría midiendo la platina sino la cuantización del generador. El ajuste es iterativo y
se siembra en la mitad central de la pata.

Al final, la tabla que compara los tres métodos sobre la misma corrida.

In [ ]:
# =============================================================================
#  CELDA 4 - metodo C: el tramo lineal es donde una RECTA todavia describe
#            a la posicion medida
# =============================================================================
# A y B preguntan "¿la velocidad (o el error) esta en su meseta?". Este pregunta
# directamente lo que importa: ¿hasta donde la posicion es una recta en el
# tiempo? Se ajusta una recta y se acepta el tramo mientras el residuo no supere
# un criterio que NO se elige a mano:
#   - sigma de sensado, de la zona quieta del final de la grabacion. Se toma la
#     mayor entre la estimacion robusta mediana(|dx|)/(0.6745*sqrt(2)) y el
#     desvio estandar de esas diferencias (la mediana satura en los 0.1 nm de
#     resolucion del recorder);
#   - dx/2: la comandada es una ESCALERA de dx nm, asi que aunque el seguimiento
#     fuera perfecto el residuo oscilaria +-dx/2 por construccion.
# El criterio es el mayor de los dos: por debajo de dx/2 no se estaria midiendo
# la platina sino la cuantizacion del generador.
K = 3.0          # cuantas sigmas se toleran


def criterio_residuo(r, k=K):
    """Cuanto residuo se tolera, en nm, y de donde sale."""
    n = ruido(r)
    sigma = max(n["mediana_dif_nm"] / (0.6745 * np.sqrt(2)), n["sigma_nm"])
    return max(k * sigma, r["dx_nm"] / 2), sigma, r["dx_nm"] / 2


def tramo_por_recta(r, x, idx, criterio_nm, n_iter=8):
    """Tramo contiguo mas largo de una pata donde una recta ajusta dentro del criterio.

    Semilla: la mitad central de la pata. Se ajusta, se miran los residuos de
    toda la pata, se acepta el tramo contiguo mas largo con |residuo| <=
    criterio, y se vuelve a ajustar hasta que deja de moverse.
    """
    t = r["t_ms"]
    sel = idx[len(idx) // 4: 3 * len(idx) // 4]
    res = None
    for _ in range(n_iter):
        p = np.polyfit(t[sel], x[sel], 1)
        res = (x[idx] - np.polyval(p, t[idx])) * 1e3
        nuevo = idx[tramo_mas_largo(np.abs(res) <= criterio_nm)]
        if len(nuevo) < 3 or np.array_equal(nuevo, sel):
            break
        sel = nuevo
    return sel, res


def tramos_por_recta(r, x, k=K):
    """Metodo C sobre las dos patas. Devuelve la segmentacion y el residuo."""
    crit, sigma, escalera = criterio_residuo(r, k)
    seg, residuo = {}, np.full(len(r["t_ms"]), np.nan)
    for cual, idx in patas(r).items():
        sel, res = tramo_por_recta(r, x, idx, crit)
        seg[cual], residuo[idx] = sel, res
    return seg, residuo, crit, sigma, escalera


seg_C, residuo_C, crit, sigma, escalera = tramos_por_recta(r, r["x_med"])
res_C = resumen(r, seg_C, f"METODO C · recta sobre la posicion medida, |res| ≤ {crit:.1f} nm")
reporte(res_C, r)
print(f"   criterio = max(3σ, dx/2) = max({K*sigma:.2f}, {escalera:.2f}) = {crit:.2f} nm")

print(f"\n{'metodo':<26}{'tramo ida':>16}{'puntos':>9}{'solapamiento':>15}"
      f"{'desfasaje':>12}{'tau':>9}")
for nombre, rr in (("A · velocidad comandada", res_A), ("B · error del controlador", res_B),
                   ("C · recta sobre la medida", res_C)):
    d = rr["tramos"]["ida"]
    print(f"{nombre:<26}{d['t0']:7.1f}..{d['t1']:6.1f} ms{rr['puntos_totales']:>9.0f}"
          f"{rr['solapamiento_nm']:>12.0f} nm{rr['desfasaje_nm']:>+9.0f} nm{rr['tau_ms']:>7.1f} ms")

fig, ax = fig_tramo(r, res_C, residuo_C, "residuo a la recta [nm]",
                    bandas=[(crit, "0.4"), (-crit, "0.4")],
                    titulo=f"(a) {CORRIDA}: tramo lineal por el residuo a la recta")
ax[1].set_ylim(-4 * crit, 4 * crit)
ax[1].set_title(f"(b) el criterio: |residuo| ≤ {crit:.1f} nm", fontsize=8.5, loc="left")
plt.show()

fig_solapamiento(r, res_C, "(c) solapamiento segun el metodo C")
plt.show()

## Celda 5 — el mismo método C, en cualquier otra medición

Para ver si lo de la celda 4 es general o era esa corrida. **Se cambia `IDX` por el número
de la columna `#` de la celda 1** y se vuelve a correr: se muestra sólo lo necesario para
juzgar el ajuste —

- **(a)** la posición con la **ida y la vuelta marcadas** y el tramo lineal de cada una
  encima;
- **(b)** el **residuo a la recta** dentro de cada tramo, con el criterio dibujado;
- **(c)** los dos residuos superpuestos desde su propio arranque, que es la forma directa
  de comparar la ida con la vuelta: en varias corridas la ida muestra una ondulación
  regular (la escalera filtrada) y la vuelta una deriva distinta.

Vale la pena mirar números altos (amplitudes de 3, 10 y 30 µm) y bajos (0.05 y 0.1 µm,
donde `dx/2` cae por debajo del ruido y el tramo se corta en pocos ms).

In [ ]:
# =============================================================================
#  CELDA 5 - el mismo metodo C, en la medicion que se quiera
# =============================================================================
# Cambiar IDX por el numero (columna "#") que tiene la corrida en la tabla de la
# celda 1. Se muestra solo lo que hace falta para juzgar el ajuste: la posicion
# con la ida y la vuelta marcadas, y el residuo a la recta de cada una.
IDX = 64         # <<<<<<  cambiar este numero

fila = RAMPAS.loc[IDX]
ri = cargar(fila.stem)
seg_i, residuo_i, crit_i, sigma_i, escalera_i = tramos_por_recta(ri, ri["x_med"])
res_i = resumen(ri, seg_i, f"METODO C · #{IDX}")

print(f"#{IDX}  {fila.stem}")
print(f"   amplitud {ri['amplitud_um']} um · n_total {ri['n_total']} · WTR {ri['wtr']} · "
      f"RTR {ri['rtr']} · SUD {ri['sud']} · dx {ri['dx_nm']:.2f} nm · v {ri['v_um_s']:.2f} um/s")
print(f"   criterio |residuo| ≤ {crit_i:.2f} nm = max(3σ = {3*sigma_i:.2f}, "
      f"dx/2 = {escalera_i:.2f})")
reporte(res_i, ri)

fig, ax = fig_tramo(ri, res_i, residuo_i, "residuo a la recta [nm]",
                    bandas=[(crit_i, "0.4"), (-crit_i, "0.4")],
                    titulo=f"(a) #{IDX} {fila.stem}")
ax[1].set_ylim(-4 * crit_i, 4 * crit_i)
ax[1].set_title("(b) residuo a la recta dentro de cada tramo lineal", fontsize=8.5,
                loc="left")
plt.show()

# las dos patas superpuestas desde su propio arranque: asi se compara la forma
# del residuo de la ida con la de la vuelta
fig, ax = plt.subplots(figsize=(7.8, 2.2))
for cual, d in res_i["tramos"].items():
    i = d["i"]
    ax.plot(ri["t_ms"][i] - ri["t_ms"][i[0]], residuo_i[i], lw=0.7, color=COL[cual],
            label=f"{cual}  (rms {np.sqrt(np.nanmean(residuo_i[i]**2)):.2f} nm)")
for s in (+1, -1):
    ax.axhline(s * crit_i, color="0.4", ls="--", lw=0.8)
ax.set_xlabel("t desde el arranque del tramo lineal [ms]")
ax.set_ylabel("residuo [nm]")
ax.legend(frameon=False, fontsize=8)
ax.set_title("(c) ida y vuelta superpuestas: ¿tienen la misma forma?", fontsize=8.5,
             loc="left")
plt.show()

## Celda 6 — WTR: la perilla de la velocidad

Con la amplitud y `n_total` fijos, `dx` (el tamaño del pixel) no cambia: **WTR es, literal,
la perilla de la velocidad**. Se elige la familia de corridas que sólo difieren en WTR y se
mide, con el método C, una sola cosa en dos paneles:

- **(a)** el **desfasaje** contra la velocidad, con la predicción `2·v·tau` superpuesta. Es
  la verificación de que el desfasaje no es un artefacto del criterio sino el retardo del
  lazo;
- **(b)** cuántos **puntos comunes** a la ida y a la vuelta sobreviven.

Todo lo demás (fracción lineal, rendimiento, residuos) salió de acá: era información que no
cambiaba la conclusión.

In [ ]:
# =============================================================================
#  CELDA 6 - una sola pregunta: que pasa cuando se cambia la VELOCIDAD (WTR)
# =============================================================================
# WTR = cuantos ciclos de servo (40 us) dura cada punto de la wave. Con la
# amplitud y n_total fijos, dx (el tamano del pixel) no cambia: WTR es, literal,
# la perilla de la velocidad. Se comparan corridas que solo difieren en WTR.
# Todo lo que sigue usa el METODO C (el residuo a la recta sobre la posicion
# medida), que es el que mide la platina y no el generador.

def metricas(stem, tol=TOL, tol_e=TOL_E, k=K):
    """Una fila por corrida: los tres metodos, reducidos a lo que se usa."""
    r = cargar(stem)
    f = {"stem": stem, "wtr": r["wtr"], "sud": r["sud"], "n_total": r["n_total"],
         "amplitud_um": r["amplitud_um"], "dx_nm": r["dx_nm"], "v_um_s": r["v_um_s"],
         "wave_ms": r["n_total"] * r["dwell_us"] / 1e3}

    v = derivar(r["t_ms"], r["x_cmd"], r["muestras_por_punto"])
    ra = resumen(r, meseta_por_tolerancia(r, v, tol)[0])
    e = suavizar(r["err_nm"], r["muestras_por_punto"])
    rb = resumen(r, meseta_por_tolerancia(r, e, tol_e)[0])
    seg_c, residuo, crit, *_ = tramos_por_recta(r, r["x_med"], k)
    rc = resumen(r, seg_c)

    for etq, rr in (("A", ra), ("B", rb), ("C", rc)):
        d = rr["tramos"].get("ida", {})
        f[f"dur_ida_{etq}_ms"] = d.get("dur_ms", np.nan)
        f[f"t_antes_{etq}_ms"] = d.get("t_antes_ms", np.nan)
        f[f"comunes_{etq}"] = rr.get("puntos_comunes", np.nan)
        f[f"solap_{etq}_nm"] = rr.get("solapamiento_nm", np.nan)
        f[f"desf_{etq}_nm"] = rr.get("desfasaje_nm", np.nan)
        f[f"tau_{etq}_ms"] = rr.get("tau_ms", np.nan)
    f["inversion_ms"] = rc.get("t_inversion_ms", np.nan)
    f["criterio_nm"] = crit
    i = rc["tramos"].get("ida", {}).get("i", np.array([], int))
    f["residuo_rms_nm"] = float(np.sqrt(np.nanmean(residuo[i] ** 2))) if len(i) else np.nan
    return f


# la familia: misma amplitud, mismos puntos y mismo SUD; solo cambia WTR.
# Se elige el grupo que barra mas valores de WTR.
fam = RAMPAS.groupby(["amplitud_um", "n_total", "sud"]).wtr.nunique().idxmax()
sel_wtr = (RAMPAS.query("amplitud_um == @fam[0] and n_total == @fam[1] and sud == @fam[2]")
           .drop_duplicates("wtr").sort_values("wtr"))
print(f"amplitud {fam[0]} um · n_total {fam[1]} · SUD {fam[2]} · WTR = {list(sel_wtr.wtr)}\n")

M_WTR = pd.DataFrame([metricas(s) for s in sel_wtr.stem])
display(M_WTR[["wtr", "v_um_s", "dur_ida_C_ms", "t_antes_C_ms", "comunes_C",
               "desf_C_nm", "tau_C_ms", "residuo_rms_nm"]].round(2))

# tau del conjunto: la mediana de los tau de cada corrida (desfasaje / 2v)
v, d = M_WTR.v_um_s.values, M_WTR.desf_C_nm.abs().values
tau = float(np.nanmedian(M_WTR.tau_C_ms))

fig, ax = plt.subplots(1, 2, figsize=(8.6, 3.0))
ax[0].plot(v, d, "o", color="k", label="medido")
vv = np.linspace(0, v.max() * 1.1, 50)
ax[0].plot(vv, 2 * vv * tau * 1e-3 * 1e3, "--", color=COL["vuelta"],
           label=f"2·v·tau con tau = {tau:.1f} ms (mediana)")
ax[0].set_xlabel("v [um/s]")
ax[0].set_ylabel("desfasaje ida↔vuelta [nm]")
ax[0].legend(frameon=False, fontsize=8)
ax[0].set_title("(a) el desfasaje es 2·v·tau", fontsize=9, loc="left")

ax[1].plot(M_WTR.wtr, M_WTR.comunes_C, "o-", color=COL["comun"])
ax[1].set_xticks(M_WTR.wtr)
ax[1].set_xlabel("WTR [ciclos de servo por punto]")
ax[1].set_ylabel("puntos comunes a ida y vuelta")
ax[1].set_title("(b) cuantos puntos quedan utilizables", fontsize=9, loc="left")
plt.tight_layout()
plt.show()

print(f"tau = desfasaje/2v va de {M_WTR.tau_C_ms.min():.1f} a "
      f"{M_WTR.tau_C_ms.max():.1f} ms en esta familia (mediana {tau:.1f} ms). El punto\n"
      f"mas rapido se cae de la recta: ahi el tramo lineal dura {M_WTR.dur_ida_C_ms.min():.0f} ms "
      f"y el ajuste ya no tiene de donde agarrarse.")
print("El desfasaje crece lineal con v, asi que si el scan tiene que cerrar ida y vuelta\n"
      "sobre la misma grilla, la tolerancia en nm es la que fija WTR.")

## Celda 7 — SUD: los puntos que se gastan en las puntas

SUD son los puntos que el generador gasta en cada punta acelerando y frenando: se pagan en
tiempo y no se usan para medir. Dos preguntas y nada más:

- **(a)** ¿achican el **tiempo de inversión** (frenar la ida + arrancar la vuelta) y el
  tiempo perdido antes del tramo lineal?
- **(b)** ¿queda **más o menos tramo útil**?

Trampa a tener presente al leer la tabla: con SUD > 0 el generador reparte los `n_total`
puntos de forma **no uniforme** — gasta puntos en las puntas y los del medio quedan más
separados — así que `dx` (y con él la velocidad) sube aunque no se haya tocado WTR. Por eso
van las dos columnas juntas: `dx_nm` es la precisión espacial y `comunes_C` los puntos
útiles que sobreviven.

In [ ]:
# =============================================================================
#  CELDA 7 - una sola pregunta: para que sirven los puntos de punta (SUD)
# =============================================================================
# SUD = cuantos puntos de cada punta gasta el generador acelerando y frenando.
# Se pagan en tiempo y no se usan para medir. Las dos preguntas, y nada mas:
#   (a) ¿achican el tiempo de inversion (frenar la ida + arrancar la vuelta)?
#   (b) ¿queda mas o menos tramo util?
# Trampa a tener presente: con SUD > 0 el generador reparte los n_total puntos
# de forma no uniforme, asi que dx (y con el la velocidad) sube aunque no se
# haya tocado WTR. Por eso la tabla lleva las dos columnas juntas.

fam_s = RAMPAS.groupby(["amplitud_um", "n_total", "wtr"]).sud.nunique().idxmax()
sel_sud = (RAMPAS.query("amplitud_um == @fam_s[0] and n_total == @fam_s[1] "
                        "and wtr == @fam_s[2]")
           .drop_duplicates("sud").sort_values("sud"))
print(f"amplitud {fam_s[0]} um · n_total {fam_s[1]} · WTR {fam_s[2]} · "
      f"SUD = {list(sel_sud.sud)}\n")

M_SUD = pd.DataFrame([metricas(s) for s in sel_sud.stem])
display(M_SUD[["sud", "dx_nm", "v_um_s", "dur_ida_C_ms", "t_antes_C_ms",
               "inversion_ms", "comunes_C", "desf_C_nm"]].round(2))

fig, ax = plt.subplots(1, 2, figsize=(8.6, 3.0))
ax[0].plot(M_SUD.sud, M_SUD.inversion_ms, "o-", color="k", label="inversion")
ax[0].plot(M_SUD.sud, M_SUD.t_antes_C_ms, "s--", color="0.55",
           label="perdido antes del tramo de ida")
ax[0].set_xlabel("SUD [puntos por punta]")
ax[0].set_ylabel("[ms]")
ax[0].legend(frameon=False, fontsize=8)
ax[0].set_title("(a) el tiempo que se paga en las puntas", fontsize=9, loc="left")

ax[1].plot(M_SUD.sud, M_SUD.comunes_C, "o-", color=COL["comun"])
ax[1].set_xlabel("SUD [puntos por punta]")
ax[1].set_ylabel("puntos comunes a ida y vuelta")
ax[1].set_title("(b) cuantos puntos quedan utilizables", fontsize=9, loc="left")
plt.tight_layout()
plt.show()

mejor = M_SUD.loc[M_SUD.comunes_C.idxmax()]
print(f"mas puntos comunes: SUD = {mejor.sud:.0f} ({mejor.comunes_C:.0f} puntos, "
      f"tramo de ida de {mejor.dur_ida_C_ms:.1f} ms, inversion {mejor.inversion_ms:.1f} ms)")

## Celda 8 — la tabla

Un renglón por medición, con lo que salió de cada ajuste: duración del tramo lineal, puntos
comunes a las dos pasadas, solapamiento en nm, desfasaje, el `tau` que sale de él
(`desfasaje / 2v`), el `tau` que da el método B (`|e|/v`) y el residuo rms a la recta.

El **desfasaje no es el mismo en todas** — crece con la velocidad, va de ~150 nm a ~8.8 µm
— pero el `tau` que sale de dividirlo por `2v` **sí** se repite corrida a corrida. Ése es el
número del cuaderno, y coincide con el que da la meseta del error, que es una medición
completamente distinta.

El resumen final descarta lo que no se puede medir: amplitudes menores a 1 µm (el desfasaje
queda del orden del ruido) y las corridas donde la wave no entra entera en la ventana
grabada, que se detectan solas porque la ida y la vuelta no llegan a solaparse.

In [ ]:
# =============================================================================
#  CELDA 8 - la tabla: un renglon por medicion, con lo que se saco de cada ajuste
# =============================================================================
# Se corren los tres metodos sobre TODAS las corridas de la celda 1. Columnas:
#   dur_ida    duracion del tramo lineal de la ida (metodo C)
#   comunes    puntos de la wave comunes a la ida y a la vuelta
#   solap      lo mismo, en nm de recorrido
#   desfasaje  cuanto se corre la vuelta respecto de la ida en el mismo x
#   tau_C      desfasaje / 2v   ·   tau_B  la meseta del error / v
#   residuo    rms a la recta dentro del tramo de ida

TABLA = pd.DataFrame([metricas(s) for s in RAMPAS.stem])
TABLA.insert(0, "#", RAMPAS.index)

vista = TABLA[["#", "amplitud_um", "n_total", "wtr", "sud", "dx_nm", "v_um_s",
               "dur_ida_C_ms", "comunes_C", "solap_C_nm", "desf_C_nm",
               "tau_C_ms", "tau_B_ms", "residuo_rms_nm"]].round(2)
vista.columns = ["#", "A[um]", "n", "WTR", "SUD", "dx[nm]", "v[um/s]", "dur_ida[ms]",
                 "comunes", "solap[nm]", "desfasaje[nm]", "tau_C[ms]", "tau_B[ms]",
                 "residuo[nm]"]
display(vista.set_index("#"))

# el desfasaje no es el mismo en todas (crece con v), pero el tau que sale de el
# si tiene que serlo. Se resume sobre las corridas donde la medida tiene sentido:
# excursiones de 1 um o mas (con 0.1 um el desfasaje es del orden del ruido) y
# las dos pasadas con algo en comun (si la wave no entra entera en la ventana
# grabada, la vuelta queda cortada y no hay solapamiento).
valida = lambda col: TABLA[(TABLA.amplitud_um >= 1.0) & (TABLA[col] > 0)]
for nombre, col_tau, col_solap in (("tau del desfasaje (C)", "tau_C_ms", "solap_C_nm"),
                                   ("tau del error (B)", "tau_B_ms", "solap_B_nm")):
    x = valida(col_solap)[col_tau].dropna()
    print(f"{nombre:<24}: mediana {x.median():5.2f} ms   MAD "
          f"{(x - x.median()).abs().median():4.2f} ms   rango {x.min():5.2f} .. "
          f"{x.max():5.2f} ms   ({len(x)} corridas validas)")

buenas = valida("solap_C_nm")
print(f"\ndesfasaje = 2·v·tau: va de {buenas.desf_C_nm.min():.0f} a "
      f"{buenas.desf_C_nm.max():.0f} nm segun la velocidad (de "
      f"{buenas.v_um_s.min():.1f} a {buenas.v_um_s.max():.1f} um/s); "
      f"lo que se repite es el tau.")